# Code Overlay Validation Lease Design

**Status:** Written specification for review; architecture direction approved on 2026-08-26  
**Design epic:** `bd-2r17`  
**Optimization evidence:** `sol_9aa374fea73d49ed` (Z3 4.16.0, SAT, lexicographic optimum, termination complete)  
**Authoritative artifact:** this notebook  
**Formal profiles:** `relational_lia@1`, `sequence_trace@1` — implemented, inline annotations, Z3 `qf_lia_bool_int_enum`

## Executive decision

Replace repeated per-layer Git/worktree validation with a **per-worktree validation manager** that lends each complete `code_*` MCP request an immutable overlay-generation lease guarded by synchronous start/end fsmonitor token fences.

A response may escape only when:

`start_token == generation_token == end_token`

Otherwise the result is discarded and retried once; continued churn or any fsmonitor uncertainty routes to exact fail-closed validation.

TTL is retained only for memory eviction and periodic defense-in-depth audits. It is never a freshness authority.

## Problem and measured evidence

The generation/index work is no longer the warm-path bottleneck:

| Phase | Measured warm p50 |
|---|---:|
| Generation query | 0.003–0.010 ms |
| Git freshness observation | about 34 ms |
| Response-file metadata/OID analysis | about 8.2 ms |
| Complete warm MCP request | about 128–132 ms |
| Repeated/unattributed validation remainder | about 85–89 ms |

Current validation is layered across `code_graph_backend_response_with_refresh`, `prepare_stable_overlay_for_worktree`, `snapshot_with_runtime`, `authoritative_overlay_identity`, and `GraphResponseMetadata::analyze_source_inner`.

The graph index used for design orientation reported stale response-file OIDs, so exact line-level claims are grounded in the current worktree inspection performed for `bd-z2d7`; graph search remains useful for symbol boundaries, not byte freshness.

## Goals, non-goals, and terms

### Goals

1. Preserve the exact invariant that no response observes a stale or mixed overlay generation.
2. Remove full Git status and response-file filesystem scans from the healthy warm path.
3. Pin one immutable generation across all nested operations in a logical MCP request.
4. Update generations incrementally from changed paths and coalesce concurrent rebuilds.
5. Retain broad compatibility through exact fallback.
6. Release only from correctness oracles and measured cross-project latency.

### Non-goals

- No TTL-based correctness shortcut.
- No asynchronous watcher queue as the sole freshness authority.
- No changes to query semantics, ranking, stable IDs, or output ordering.
- No always-on background indexing requirement.
- No promise that unsupported/non-local filesystems meet the fast-path target.
- No dependency on undocumented Git IPC until a probe proves a stable seam.

### Terms

- **Token fence:** synchronous fsmonitor observation to a monotonic token.
- **Generation:** immutable visible indexes plus base/worktree identity and content-OID manifest.
- **Lease:** request-scoped generation `Arc` plus validation token/epoch.
- **Exact fallback:** the conservative Git/index/file oracle, consolidated to one observation per request attempt.

## Fast-path eligibility

The token route is enabled only when every capability is proven for the current worktree and release configuration. Any failed condition selects exact fallback; no partial fast path exists.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-VALIDATION-ELIGIBILITY
@type ValidationRoute = enum[token_lease, exact_fallback]
@input release_enabled: Bool
@input builtin_supported: Bool
@input local_filesystem: Bool
@input watcher_healthy: Bool
@input synchronous_token_fence: Bool
@output status: ValidationRoute
@requires PRE: true`"]

    FAST["`@branch FAST
@when release_enabled and builtin_supported and local_filesystem and watcher_healthy and synchronous_token_fence
@ensures FAST_ROUTE: status = token_lease`"]

    FALLBACK["`@branch FALLBACK
@when not (release_enabled and builtin_supported and local_filesystem and watcher_healthy and synchronous_token_fence)
@ensures FALLBACK_ROUTE: status = exact_fallback`"]

    CHECK["`@verify ELIGIBILITY_DETERMINISTIC: prove determinism
@verify ELIGIBILITY_COVERAGE: prove partition_coverage
@verify ELIGIBILITY_EXCLUSIVE: prove partition_exclusive
@verify ELIGIBILITY_STATUSES: witness each status
@verify FAST_REACHABLE: witness branch FAST
@verify FALLBACK_REACHABLE: witness branch FALLBACK`"]

    SPEC --> FAST --> CHECK
    SPEC --> FALLBACK --> CHECK

## Architecture and ownership

### `WorktreeValidationManager`

One entry per `(canonical_worktree, graph_store_hash, indexed_base_oid, overlay_policy)`. It owns the current immutable generation, accepted token, monotonic epoch, health state, and one single-flight rebuild slot.

### `TokenFenceSource`

Returns `Stable { token, changed_paths }`, `Reset { reason }`, or `Unavailable { reason }`. The first implementation uses only a supported synchronous Git/fsmonitor query. If the Phase 0 probe cannot prove such a seam, the token route remains probe-only and the consolidated exact observer ships instead.

### `OverlayGenerationBuilder`

Consumes the previous generation and exact changed paths, preserves structurally shared chunks for unchanged files, rebuilds the changed dependency closure, and publishes atomically after identities agree.

### `ValidationLease`

Contains the compatibility key, start token, manager epoch, `Arc<OverlayGeneration>`, generation identity/content manifest, and retry attempt. All nested `code_*` operations use its pinned client and perform no filesystem or request-time config reads.

### Response metadata

Response paths and content OIDs come from the pinned generation manifest. Memoization is keyed by `(generation_identity, response_file_set_fingerprint)` and trusted only while the lease epoch is stable. The healthy path performs no response-file stat/hash scan.

## Healthy request protocol

This is the only commit-producing token fast path. The compiler verifies all rendered messages have formal bindings and that a consistent ordered trace exists. Retry and fallback are governed by the response gate.

In [ ]:
sequenceDiagram
    participant Client
    participant Manager
    participant Fsmonitor
    participant Generation
    participant Handler

    Note over Client,Handler: @spec CODE-OVERLAY-TOKEN-LEASE-PROTOCOL<br/>@input start_token: Int<br/>@input generation_token: Int<br/>@input end_token: Int<br/>@requires CONSISTENT_FAST_PATH: start_token = generation_token and generation_token = end_token

    Client->>Manager: acquire validation lease
    Note over Client,Manager: @message ACQUIRE<br/>@from Client<br/>@to Manager<br/>@event acquire validation lease<br/>@order 1<br/>@when true<br/>@ensures ACQUIRE_SENT: true

    Manager->>Fsmonitor: synchronous start fence
    Note over Manager,Fsmonitor: @message START_FENCE<br/>@from Manager<br/>@to Fsmonitor<br/>@event synchronous start fence<br/>@order 2<br/>@when true<br/>@ensures START_OBSERVED: start_token = generation_token

    Manager->>Generation: pin immutable generation
    Note over Manager,Generation: @message PIN_GENERATION<br/>@from Manager<br/>@to Generation<br/>@event pin immutable generation<br/>@order 3<br/>@when start_token = generation_token<br/>@ensures PIN_MATCHES_START: generation_token = start_token

    Manager->>Handler: execute code query
    Note over Manager,Handler: @message EXECUTE_QUERY<br/>@from Manager<br/>@to Handler<br/>@event execute code query<br/>@order 4<br/>@when start_token = generation_token<br/>@ensures EXECUTES_PINNED: true

    Manager->>Fsmonitor: synchronous end fence
    Note over Manager,Fsmonitor: @message END_FENCE<br/>@from Manager<br/>@to Fsmonitor<br/>@event synchronous end fence<br/>@order 5<br/>@when true<br/>@ensures END_OBSERVED: end_token = generation_token

    Manager-->>Client: commit exact response
    Note over Client,Manager: @message COMMIT_RESPONSE<br/>@from Manager<br/>@to Client<br/>@event commit exact response<br/>@order 6<br/>@when start_token = generation_token and generation_token = end_token<br/>@ensures COMMIT_SAFE: start_token = end_token

    Note over Client,Handler: @verify TOKEN_LEASE_TRACE: prove sequence_protocol

## Response gate and race handling

The handler never declares its own result safe.

- Equal start, generation, and end tokens: commit.
- First mismatch: discard, incorporate changes, and retry once.
- Second mismatch: run one consolidated exact observation.
- Exact observation failure: return an error, never the prior result.

This partition makes “stale-but-fast” unrepresentable.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-RESPONSE-GATE
@type ResponseAction = enum[commit, retry, exact_fallback, error]
@input start_token: Int
@input generation_token: Int
@input end_token: Int
@input retry_available: Bool
@input exact_validation_succeeds: Bool
@output status: ResponseAction
@requires PRE: true`"]

    COMMIT["`@branch COMMIT
@when start_token = generation_token and generation_token = end_token
@ensures COMMIT_STATUS: status = commit`"]

    RETRY["`@branch RETRY
@when not (start_token = generation_token and generation_token = end_token) and retry_available
@ensures RETRY_STATUS: status = retry`"]

    FALLBACK["`@branch FALLBACK
@when not (start_token = generation_token and generation_token = end_token) and not retry_available and exact_validation_succeeds
@ensures FALLBACK_STATUS: status = exact_fallback`"]

    ERROR["`@branch ERROR
@when not (start_token = generation_token and generation_token = end_token) and not retry_available and not exact_validation_succeeds
@ensures ERROR_STATUS: status = error`"]

    CHECK["`@verify RESPONSE_DETERMINISTIC: prove determinism
@verify RESPONSE_COVERAGE: prove partition_coverage
@verify RESPONSE_EXCLUSIVE: prove partition_exclusive
@verify RESPONSE_STATUSES: witness each status`"]

    SPEC --> COMMIT --> CHECK
    SPEC --> RETRY --> CHECK
    SPEC --> FALLBACK --> CHECK
    SPEC --> ERROR --> CHECK

## Cache, invalidation, and concurrency

The manager cache is key-based, not time-based. An unchanged compatibility key and token retain the generation indefinitely, subject only to memory-pressure eviction. The token is version state, not part of the lookup key.

When a token changes, the builder computes the dependency closure, structurally shares unchanged indexes, publishes a new immutable generation atomically, and increments the manager epoch. Older leases remain readable but cannot commit unless their end fence still matches.

Concurrency contracts:

- One rebuild per compatibility key; concurrent callers await the same result.
- Different worktrees and graph-store hashes never share token/generation state.
- Readers do not hold the rebuild lock while handlers execute.
- Publication is monotonic by epoch.
- Waiter cancellation does not cancel a rebuild needed by other waiters.
- Reset/overflow invalidates token trust before a new lease is issued.

TTL may evict idle entries or schedule defense-in-depth exact audits. It never changes a freshness decision.

## Failure and compatibility matrix

| Condition | Required behavior |
|---|---|
| Feature Off | Existing exact path |
| Release gate not met | Probe-only; exact path serves requests |
| Built-in fsmonitor unsupported | Exact fallback |
| Non-local/unsupported filesystem | Exact fallback |
| Daemon unhealthy, reset, overflow, malformed response | Invalidate token state; exact fallback |
| Start token differs from generation | Incremental rebuild before handler |
| End token differs after handler | Discard and retry once |
| Token changes again during retry | Consolidated exact validation |
| Exact validation fails | Error; never stale/cached response |
| Concurrent identical refreshes | Single-flight rebuild |
| Compatibility key differs | Isolated manager entry |
| Manager entry evicted | Cold exact/bootstrap path |

Fallback is a correctness mode, not an error-recovery cache. Its observation is shared across snapshot identity, postflight safety, and metadata so it runs once per request attempt.

## Release gate

The token implementation remains probe-only until semantic and latency gates pass. A 9 ms two-fence budget leaves about 1 ms for the already-measured generation query, routing, and serialization within the 10 ms complete warm-request target.

In [ ]:
flowchart TD
    SPEC["`@spec CODE-OVERLAY-VALIDATION-RELEASE
@type ReleaseDecision = enum[release, probe_only]
@input race_tests_pass: Bool
@input fallback_tests_pass: Bool
@input overflow_tests_pass: Bool
@input cross_project_pass: Bool
@input full_warm_p95_us: Int
@input two_fence_p95_us: Int
@output status: ReleaseDecision
@requires NONNEGATIVE_LATENCY: full_warm_p95_us >= 0 and two_fence_p95_us >= 0`"]

    RELEASE["`@branch RELEASE
@when race_tests_pass and fallback_tests_pass and overflow_tests_pass and cross_project_pass and full_warm_p95_us <= 10000 and two_fence_p95_us <= 9000
@ensures RELEASE_STATUS: status = release`"]

    PROBE["`@branch PROBE
@when not (race_tests_pass and fallback_tests_pass and overflow_tests_pass and cross_project_pass and full_warm_p95_us <= 10000 and two_fence_p95_us <= 9000)
@ensures PROBE_STATUS: status = probe_only`"]

    CHECK["`@verify RELEASE_DETERMINISTIC: prove determinism
@verify RELEASE_COVERAGE: prove partition_coverage
@verify RELEASE_EXCLUSIVE: prove partition_exclusive
@verify RELEASE_STATUSES: witness each status`"]

    SPEC --> RELEASE --> CHECK
    SPEC --> PROBE --> CHECK

## TDD and evaluation contract

Implementation follows committed RED → GREEN with PRE and POST performance evidence.

### Correctness RED tests

1. A change between start/end fences prevents the first result from escaping.
2. A queued change immediately before the start fence is observed before leasing.
3. A second race during retry routes to exact fallback.
4. Reset, overflow, malformed token, and unhealthy watcher fail closed.
5. Unsupported/non-local filesystems never enter the token route.
6. Generation-derived response metadata matches the exact oracle.
7. Different compatibility keys never cross-seed.
8. Concurrent requests trigger one rebuild and pin one published generation.
9. Unchanged data retains `Arc::ptr_eq` structural sharing.
10. Feature Off preserves current output bytes.

### Healthy warm structural assertions

- full Git status observations: **0**
- response-file stat/hash/OID scans: **0**
- generation rebuilds: **0**
- legacy merge/filter/sort/dedup stages: **0**
- synchronous token fences: exactly **2**

### Benchmark matrix

Run at least 30 repetitions on small, medium, and large repositories for unchanged warm, one tracked change, one untracked change, rename/delete, concurrent identical requests, fsmonitor reset/overflow, exact fallback, and complete MCP requests.

Record p50/p95 per phase. Release requires complete warm MCP p95 ≤ 10 ms and two-fence p95 ≤ 9 ms on all supported local test projects, with exact-oracle equivalence for every response.

## Migration and rollout

1. **Supported-seam probe:** measure synchronous Git fsmonitor token/query behavior and verify delivery, reset, overflow, rename, delete, untracked, submodule, and reopen behavior. This phase may reject the token seam.
2. **Consolidate exact validation:** share one request-scoped observation across snapshot, postflight, and metadata; derive metadata from the pinned generation.
3. **Manager behind Auto/probe:** add keyed manager, immutable leases, incremental refresh, single-flight publication, and telemetry, while exact mode still serves.
4. **Release healthy local fast path:** enable only for capability-qualified worktrees after the formal release gate passes.

A runtime/config kill switch forces exact mode. Query semantics and generation representation remain unchanged, so rollback needs no cache migration. Telemetry records route, retry/reset/rebuild/fallback counts and phase latency without repository paths.

## Implementation boundaries and dependencies

1. **Probe and oracle contract** — token seam, fixtures, phase benchmark; no production routing.
2. **Shared exact observation** — consolidate repeated validation and generation-derived metadata.
3. **Manager and lease types** — compatibility isolation, atomic publication, single-flight.
4. **Token fast path** — fences, incremental rebuild, bounded retry, exact fallback.
5. **MCP wiring** — pin one lease across every `code_*` handler and remove nested validation.
6. **Release matrix** — 30-run measurements, exact oracle, configuration decision.

Tasks 1 and 2 can begin independently. Task 3 depends on Task 2’s identity contract; Tasks 4–6 are ordered.

## Risks and fixed decisions

- No supported low-latency Git seam → stop after shared exact validation.
- Watcher queue race → synchronous fences mandatory.
- Reset/overflow ambiguity → exact fallback.
- High churn → one retry, then exact validation.
- Memory growth → LRU/budget eviction only, never freshness TTL.
- Platform variance → capability-qualified per worktree.
- Host variance → release by per-phase and complete-request p95.

Exactness outranks latency; one immutable generation covers a logical request; metadata comes from that generation; exact fallback is permanent; under 10 ms is an acceptance target, not a solver proof.

## Formal proof evidence

All native NS-Mermaid cells were profile-pinned, preflighted, executed, and persisted. Every mandatory facet matched; all proof evidence is source-fresh with zero mismatches or inconclusive obligations.

| @spec | Cell | Profile | Matched | Source hash | IR hash | Report hash |
|---|---|---|---:|---|---|---|
| `CODE-OVERLAY-VALIDATION-ELIGIBILITY` | `00000000-0000-4000-8000-000000000005` | relational_lia@1 | 7/7 | `7d97007accf2905db3b7f59ccc962d47773a0a65d137906c1724a2cd98f91a0c` | `ea794751a6469023e9c6252c009b2850d2afe8346bee30a0f40f93a5c5c9c466` | `c256aea4ac6be8f21b727f2a4ef1ae183ab5f5f149d37f6d03796fdc707861f7` |
| `CODE-OVERLAY-TOKEN-LEASE-PROTOCOL` | `00000000-0000-4000-8000-000000000008` | sequence_trace@1 | 1/1 | `da8a58633562c74956bf6677dfa00aa77676cb026728edd5d59bd610a8575ba6` | `07fbf9977af571b11e8274d6763cca07a1065317fe3b433937fa41726092dc4c` | `748083a66dfa9aff68b5a9c3f140ed0c6c19dd0a949aae33a472d6b91f4d3e05` |
| `CODE-OVERLAY-RESPONSE-GATE` | `00000000-0000-4000-8000-000000000010` | relational_lia@1 | 7/7 | `fe8aa7dd23bd6e42540bed8d878e4aa315bcc4593f0195c5ebb03b0cf88bad18` | `1e03bb485abaa750d9bd1b2563460fd092458667338f1cbb068db419b127dcae` | `c7385408dee16ebb8901d9e24853cdd5cc8f51b11b2d0b1ca4f66a6972326704` |
| `CODE-OVERLAY-VALIDATION-RELEASE` | `00000000-0000-4000-8000-000000000014` | relational_lia@1 | 5/5 | `c290f3aafd6bff84d964107107ce0bc0278c72f1c47e8e2cfd5d724732a167ce` | `9e6db6e84f551dd39b47215e8b51a22f2f582c020de2c1887e2ac0813fe281a5` | `cbce96684740498e26e16812c0a44d692caa783b6fee5322a44187234ecad879` |

Independent architecture optimization: `sol_9aa374fea73d49ed`, reloaded successfully with `git_fsmonitor_token_lease` as the complete lexicographic optimum.

## Approval boundary

This notebook is ready for written-spec review. Implementation planning must bind to these four `@spec` IDs and may not weaken exactness, synchronous fencing, fallback, or release-gate obligations without revising and re-approving this notebook.